In [39]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("aqi_data_delhi")

files = list(DATA_DIR.glob("*.csv"))

dfs = []

for file in files:
    df = pd.read_csv(file)

    # Keep track of the source file
    df["source_file"] = file.name

    dfs.append(df)

delhi = pd.concat(dfs, ignore_index=True)

print("Number of files:", len(files))
print("Rows:", len(delhi))
print(delhi.shape)

Number of files: 20
Rows: 175320
(175320, 26)


In [40]:
print(delhi.columns.tolist())
print(delhi.head())
             

['Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)', 'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)', 'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)', 'source_file']
             Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01 00:00:00         160.75        262.25       25.12        44.80   
1  2024-01-01 01:00:00         170.50        254.00       22.10        49.28   
2  2024-01-01 02:00:00         172.50        243.25       13.68        45.20   
3  2024-01-01 03:00:00         170.00        217.75       15.42        40.05   
4  2024-01-01 04:00:00         183.00        241.50       12.77        39.15   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  ...  \
0      44.22        19.60         3.82        1.5

In [41]:
station_mapping = {
    "anand_vihar": "Anand Vihar",
    "dwarka-sector_8": "Dwarka Sector 8",
    "imd_lodhi_road": "IMD Lodhi Road",
    "ito": "ITO",
    "punjabi_bagh": "Punjabi Bagh",
}

In [42]:
def get_station(filename):

    filename = filename.lower()

    for key, station in station_mapping.items():
        if key in filename:
            return station

    return "Unknown"

delhi["station"] = delhi["source_file"].apply(get_station)
print(delhi["station"].value_counts())

station
Anand Vihar        35064
Dwarka Sector 8    35064
IMD Lodhi Road     35064
ITO                35064
Punjabi Bagh       35064
Name: count, dtype: int64


In [43]:
delhi["Timestamp"] = pd.to_datetime(delhi["Timestamp"], errors="coerce")
delhi = delhi.dropna(subset=["Timestamp"])
delhi = delhi.sort_values(["station", "Timestamp"])
delhi = delhi.drop_duplicates(
    subset=["station", "Timestamp"]
)

In [44]:
rename_map = {
    "Timestamp": "timestamp",
    "PM2.5 (µg/m³)": "PM25",
    "PM10 (µg/m³)": "PM10",
    "NO (µg/m³)": "NO",
    "NO2 (µg/m³)": "NO2",
    "NOx (ppb)": "NOx",
    "NH3 (µg/m³)": "NH3",
    "SO2 (µg/m³)": "SO2",
    "CO (mg/m³)": "CO",
    "Ozone (µg/m³)": "O3",
    "Benzene (µg/m³)": "Benzene",
    "Toluene (µg/m³)": "Toluene",
    "Xylene (µg/m³)": "Xylene",
    "O Xylene (µg/m³)": "O_Xylene",
    "Eth-Benzene (µg/m³)": "Eth_Benzene",
    "MP-Xylene (µg/m³)": "MP_Xylene",
    "AT (°C)": "temperature",
    "RH (%)": "humidity",
    "WS (m/s)": "wind_speed",
    "WD (deg)": "wind_direction",
    "RF (mm)": "rainfall",
    "TOT-RF (mm)": "total_rainfall",
    "SR (W/mt2)": "solar_radiation",
    "BP (mmHg)": "pressure",
    "VWS (m/s)": "vertical_wind_speed",
}

delhi = delhi.rename(columns=rename_map)

In [45]:
coverage = (
    delhi.groupby("station")
    .agg(
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        observations=("timestamp", "count")
    )
)

print(coverage)

                     start                 end  observations
station                                                     
Anand Vihar     2022-01-01 2025-12-31 23:00:00         35064
Dwarka Sector 8 2022-01-01 2025-12-31 23:00:00         35064
IMD Lodhi Road  2022-01-01 2025-12-31 23:00:00         35064
ITO             2022-01-01 2025-12-31 23:00:00         35064
Punjabi Bagh    2022-01-01 2025-12-31 23:00:00         35064


In [46]:
delhi.head()

,timestamp,PM25,PM10,NO,NO2,NOx,NH3,SO2,CO,O3,...,humidity,wind_speed,wind_direction,rainfall,total_rainfall,solar_radiation,pressure,vertical_wind_speed,source_file,station
8784,2022-01-01 00:00:00,613.25,NaN,406.70,104.50,387.18,30.1,17.80,NaN,5.38,...,84.15,0.3,173.00,NaN,0.0,8.95,974.98,NaN,"raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H 2...",Anand Vihar
8785,2022-01-01 01:00:00,684.00,892.0,313.38,73.82,294.77,68.2,10.95,NaN,0.30,...,87.62,0.3,220.50,NaN,0.0,8.88,974.90,NaN,"raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H 2...",Anand Vihar
8786,2022-01-01 02:00:00,615.25,916.5,357.10,66.70,325.20,NaN,8.80,NaN,6.75,...,89.38,0.3,215.75,NaN,0.0,8.88,974.78,NaN,"raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H 2...",Anand Vihar
8787,2022-01-01 03:00:00,739.00,950.0,289.58,70.25,273.48,64.4,5.67,NaN,6.90,...,90.65,0.3,201.75,NaN,0.0,8.90,974.90,NaN,"raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H 2...",Anand Vihar
8788,2022-01-01 04:00:00,596.50,801.5,233.18,60.12,222.10,NaN,12.28,NaN,10.20,...,91.32,0.3,222.00,NaN,0.0,8.88,974.68,NaN,"raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H 2...",Anand Vihar


In [ ]:
delhi["city"] = "Delhi"
df = delhi.copy()
